Second try. differnt loop. only lambda hyperparameter, no lag

In [3]:
%matplotlib qt
#%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
# Use Arial everywhere (falls back if not available)
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 12,          # base font size
    "axes.titlesize": 16,     # figure/axes titles
    "axes.labelsize": 13,     # x/y labels
    "legend.fontsize": 11,    # legend text
    "xtick.labelsize": 11,    # tick labels
    "ytick.labelsize": 11,
    "figure.titlesize": 18,   # suptitle
})

In [4]:
import numpy as np
from scipy import stats

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm

# -----------------------------
# Helpers
# -----------------------------
def clean_subject_col(s):
    """
    Make sure subject is an int, whether it looks like 42 or 'sub-042'.
    """
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(int)
    # extract first integer substring
    return s.astype(str).str.extract(r'(\d+)')[0].astype(int)

def pick_col(df, candidates):
    """
    Return the first column name from `candidates` that exists in df.
    """
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of these columns found: {candidates}. Available: {list(df.columns)}")

def load_pair(eel_path, ridge_path,
              eel_candidates=("r_model", "r", "r_test", "test_r"),
              ridge_candidates=("cv_mean_test_r_star", "mean_test_r", "r_cv_star", "r_test", "ridge_nested_test_r")):
    """
    Load eelbrain + ridge files and return a merged df with columns:
    subject, eel, ridge
    """
    eel = pd.read_csv(eel_path)
    ridge = pd.read_csv(ridge_path)

    eel_sub = pick_col(eel, ["subject"])
    ridge_sub = pick_col(ridge, ["subject"])

    eel_r = pick_col(eel, eel_candidates)
    ridge_r = pick_col(ridge, ridge_candidates)

    eel = eel[[eel_sub, eel_r]].copy()
    ridge = ridge[[ridge_sub, ridge_r]].copy()

    eel.columns = ["subject", "eel"]
    ridge.columns = ["subject", "ridge"]

    eel["subject"] = clean_subject_col(eel["subject"])
    ridge["subject"] = clean_subject_col(ridge["subject"])

    merged = pd.merge(eel, ridge, on="subject", how="inner").dropna(subset=["eel", "ridge"])
    return merged

def load_pair_nested(eel_path, ridge_path,
              eel_candidates=("r_model", "r", "r_test", "test_r"),
              ridge_candidates=("cv_mean_test_r_star", "mean_test_r", "r_cv_star", "r_test")):
    """
    Load eelbrain + ridge files and return a merged df with columns:
    subject, eel, ridge
    """
    eel = pd.read_csv(eel_path)
    ridge = pd.read_csv(ridge_path)

    eel_sub = pick_col(eel, ["subject"])
    ridge_sub = pick_col(ridge, ["subject"])

    eel_r = pick_col(eel, eel_candidates)
    ridge_r = pick_col(ridge, ridge_candidates)

    eel = eel[[eel_sub, eel_r]].copy()
    ridge = ridge[[ridge_sub, ridge_r]].copy()

    eel.columns = ["subject", "eel"]
    ridge.columns = ["subject", "ridge"]

    eel["subject"] = clean_subject_col(eel["subject"])
    ridge["subject"] = clean_subject_col(ridge["subject"])

    merged = pd.merge(eel, ridge, on="subject", how="inner").dropna(subset=["eel", "ridge"])
    return merged

def plot_hist_with_gauss(ax, df, title, bins=15, show_legend=True, xlim=(-0.25, 0.45)):
    eel = df["eel"].to_numpy()
    ridge = df["ridge"].to_numpy()

    ax.hist(eel, bins=bins, density=True, alpha=0.5, label=r"mTRF ($r^{dec}$)")
    ax.hist(ridge, bins=bins, density=True, alpha=0.5, label=r"Ridge ($r^{ridge, test}$)")

    x = np.linspace(xlim[0], xlim[1], 400)

    mu_e, sig_e = norm.fit(eel)
    mu_r, sig_r = norm.fit(ridge)

    ax.plot(x, norm.pdf(x, mu_e, sig_e), linewidth=2,color="blue", label=f"mTRF fit")
    ax.plot(x, norm.pdf(x, mu_r, sig_r), linewidth=2, color="brown", label=f"Ridge fit")

    ax.axvline(0, linestyle=":", linewidth=1)
    ax.set_xlim(*xlim)
    ax.set_title(title)
    ax.set_xlabel(r"$r^{test}$")
    if show_legend:
        ax.set_ylabel("Density")
        ax.legend()
    #ax.legend(fontsize=8)
# -----------------------------
# Your paths
# -----------------------------
df_ridge_all_path  = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\match_eelbrain_cv\df_ridge_eel_nested_ALL.csv"
df_ridge_slow_path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\match_eelbrain_cv\df_ridge_eel_nested_SLOW.csv"
df_ridge_fast_path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\match_eelbrain_cv\df_ridge_eel_nested_FAST.csv"

df_eel_all_path  = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\eelbrain\fast_and_slow\ran10\decoder_all2.csv"
df_eel_slow_path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\eelbrain\ran04\decoder_slow.csv"
df_eel_fast_path = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\eelbrain\fast\ran12\decoder_fast.csv"


# -----------------------------
# Load + stats
# -----------------------------
df_combined = load_pair(df_eel_all_path,  df_ridge_all_path)   # combined
df_tonic    = load_pair(df_eel_slow_path, df_ridge_slow_path)  # tonic (slow)
df_phasic   = load_pair(df_eel_fast_path, df_ridge_fast_path)  # phasic (fast)

# -----------------------------
# Plot: 3 histograms (test only)
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
plot_hist_with_gauss(axes[0], df_phasic,   "Phasic")
plot_hist_with_gauss(axes[1], df_tonic,    "Tonic", show_legend=False)
plot_hist_with_gauss(axes[2], df_combined, "Combined", show_legend=False)
plt.tight_layout()
plt.show()

In [7]:

def fisher_z(r):
    r = np.asarray(r, float)
    r = r[np.isfinite(r)]
    r = np.clip(r, -0.999999, 0.999999)
    return np.arctanh(r)

def mean_ci_in_z_then_to_r(r, alpha=0.05):
    """
    Compute mean, SD, and CI of the mean in Fisher-z, then back-transform to r.
    Returns:
      mean_r, sd_r_from_z (delta approx), ci_r, plus z-space values.
    """
    z = fisher_z(r)
    n = len(z)
    if n < 2:
        return dict(N=n, mean_z=np.nan, sd_z=np.nan, mean_r=np.nan, sd_r=np.nan,
                    ci_z=(np.nan, np.nan), ci_r=(np.nan, np.nan))

    mean_z = float(np.mean(z))
    sd_z   = float(np.std(z, ddof=1))
    se_z   = sd_z / np.sqrt(n)
    tcrit  = stats.t.ppf(1 - alpha/2, df=n-1)

    ci_z = (mean_z - tcrit * se_z, mean_z + tcrit * se_z)

    mean_r = float(np.tanh(mean_z))
    ci_r   = (float(np.tanh(ci_z[0])), float(np.tanh(ci_z[1])))

    # SD on r scale derived from SD in z via delta method
    # dr/dz = 1 - tanh(z)^2, evaluated at mean_z
    sd_r = float((1 - mean_r**2) * sd_z)
    sd_r = r.std(ddof=1)  # empirical SD on r scale (for reference, not used in CI)

    return dict(
        N=n,
        mean_z=mean_z, sd_z=sd_z,
        mean_r=mean_r, sd_r=sd_r,
        ci_z=(float(ci_z[0]), float(ci_z[1])),
        ci_r=ci_r
    )

def bootstrap_ci_delta_r(z_eel, z_ridge, n_boot=20000, alpha=0.05, seed=0):
    """
    Bootstrap CI for delta_r = tanh(mean(z_eel)) - tanh(mean(z_ridge)).
    Resamples subjects with replacement.
    """
    rng = np.random.default_rng(seed)
    z_eel = np.asarray(z_eel, float)
    z_ridge = np.asarray(z_ridge, float)
    n = len(z_eel)
    idx = np.arange(n)

    boot = np.empty(n_boot, float)
    for b in range(n_boot):
        s = rng.choice(idx, size=n, replace=True)
        boot[b] = np.tanh(z_eel[s].mean()) - np.tanh(z_ridge[s].mean())

    lo = float(np.quantile(boot, alpha/2))
    hi = float(np.quantile(boot, 1 - alpha/2))
    return (lo, hi)

def holm_adjust(pvals):
    """Holm step-down adjustment. Returns adjusted p-values in original order."""
    pvals = np.asarray(pvals, float)
    m = len(pvals)
    order = np.argsort(pvals)
    p_sorted = pvals[order]

    adj_sorted = (m - np.arange(m)) * p_sorted
    adj_sorted = np.minimum(adj_sorted, 1.0)
    adj_sorted = np.maximum.accumulate(adj_sorted)

    adj = np.empty_like(adj_sorted)
    adj[order] = adj_sorted
    return adj


def paired_stats_fisher(df, label, alpha=0.05, do_bootstrap=True):
    """
    Descriptives (mean±SD and CI) for eel and ridge computed in z-space and back-transformed to r.
    Paired test is on delta_z = z_eel - z_ridge (two-sided).
    Also reports delta_r = tanh(mean_z_eel) - tanh(mean_z_ridge) with optional bootstrap CI.
    """
    eel_r   = df["eel"].to_numpy()
    ridge_r = df["ridge"].to_numpy()

    z_eel   = fisher_z(eel_r)
    z_ridge = fisher_z(ridge_r)

    # keep only matched finite pairs (should already be true)
    m = np.isfinite(z_eel) & np.isfinite(z_ridge)
    z_eel, z_ridge = z_eel[m], z_ridge[m]

    # Per-model mean±SD and CI in z -> r
    eel_stats   = mean_ci_in_z_then_to_r(np.tanh(z_eel), alpha=alpha)   # tanh(z)=r (clipped)
    ridge_stats = mean_ci_in_z_then_to_r(np.tanh(z_ridge), alpha=alpha)

    # Paired inference on delta_z
    diff_z = z_eel - z_ridge
    n = len(diff_z)

    t_stat, p_two = stats.ttest_1samp(diff_z, 0.0)  # paired t-test on delta_z
    # CI for mean(diff_z) in z-space
    md_z = float(np.mean(diff_z))
    sd_dz = float(np.std(diff_z, ddof=1))
    se_dz = sd_dz / np.sqrt(n)
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    ci_dz = (md_z - tcrit * se_dz, md_z + tcrit * se_dz)

    # Effect in r-units based on back-transformed means (more interpretable than tanh(delta_z))
    delta_r = float(np.tanh(z_eel.mean()) - np.tanh(z_ridge.mean()))
    ci_delta_r = (np.nan, np.nan)
    if do_bootstrap:
        ci_delta_r = bootstrap_ci_delta_r(z_eel, z_ridge, n_boot=20000, alpha=alpha, seed=0)

    print(f"\n=== {label} (Fisher-z summary) ===")
    print(f"N subjects: {n}")

    print(f"mTRF:   mean_r={eel_stats['mean_r']:.6f} ± {eel_stats['sd_r']:.6f}  "
          f"95% CI_r=[{eel_stats['ci_r'][0]:.6f}, {eel_stats['ci_r'][1]:.6f}]")
    print(f"Ridge:  mean_r={ridge_stats['mean_r']:.6f} ± {ridge_stats['sd_r']:.6f}  "
          f"95% CI_r=[{ridge_stats['ci_r'][0]:.6f}, {ridge_stats['ci_r'][1]:.6f}]")

    print(f"Paired test on Δz = z_mTRF - z_ridge: mean_Δz={md_z:.6f}  "
          f"95% CI_Δz=[{ci_dz[0]:.6f}, {ci_dz[1]:.6f}]  t={t_stat:.6f}  p={p_two:.6g}")

    print(f"Δr based on back-transformed means: {delta_r:.6f}  "
          f"{int((1-alpha)*100)}% bootstrap CI=[{ci_delta_r[0]:.6f}, {ci_delta_r[1]:.6f}]")

    return {
        "N": n,
        "eel_mean_r": eel_stats["mean_r"], "eel_sd_r": eel_stats["sd_r"], "eel_ci_r": eel_stats["ci_r"],
        "ridge_mean_r": ridge_stats["mean_r"], "ridge_sd_r": ridge_stats["sd_r"], "ridge_ci_r": ridge_stats["ci_r"],
        "delta_z_mean": md_z, "delta_z_ci": (float(ci_dz[0]), float(ci_dz[1])),
        "t": float(t_stat), "p_two": float(p_two),
        "delta_r_mean": delta_r, "delta_r_ci_boot": ci_delta_r,
    }

results = {}
results["Phasic"]   = paired_stats_fisher(df_phasic,   "PHASIC (FAST)")
results["Tonic"]    = paired_stats_fisher(df_tonic,    "TONIC (SLOW)")
results["Combined"] = paired_stats_fisher(df_combined, "COMBINED (ALL)")

# Holm correction across the three Δz paired tests
order = ["Phasic", "Tonic", "Combined"]
p_raw = [results[k]["p_two"] for k in order]
p_holm = holm_adjust(p_raw)

for k, ph in zip(order, p_holm):
    results[k]["p_holm"] = float(ph)

print("\nHolm-corrected p-values for paired Δz tests (two-sided):")
for k in order:
    print(f"{k}: p={results[k]['p_two']:.6g}, p_Holm={results[k]['p_holm']:.6g}")



=== PHASIC (FAST) (Fisher-z summary) ===
N subjects: 52
mTRF:   mean_r=0.101946 ± 0.119148  95% CI_r=[0.067881, 0.135772]
Ridge:  mean_r=0.054819 ± 0.128723  95% CI_r=[0.017823, 0.091665]
Paired test on Δz = z_mTRF - z_ridge: mean_Δz=0.047427  95% CI_Δz=[-0.000502, 0.095355]  t=1.986571  p=0.0523556
Δr based on back-transformed means: 0.047126  95% bootstrap CI=[0.000970, 0.093355]

=== TONIC (SLOW) (Fisher-z summary) ===
N subjects: 52
mTRF:   mean_r=0.071811 ± 0.115697  95% CI_r=[0.039101, 0.104366]
Ridge:  mean_r=0.054819 ± 0.128723  95% CI_r=[0.017823, 0.091665]
Paired test on Δz = z_mTRF - z_ridge: mean_Δz=0.017060  95% CI_Δz=[-0.009302, 0.043423]  t=1.299190  p=0.199722
Δr based on back-transformed means: 0.016992  95% bootstrap CI=[-0.008822, 0.042419]

=== COMBINED (ALL) (Fisher-z summary) ===
N subjects: 52
mTRF:   mean_r=0.100483 ± 0.093160  95% CI_r=[0.074179, 0.126646]
Ridge:  mean_r=0.026105 ± 0.055358  95% CI_r=[0.010641, 0.041555]
Paired test on Δz = z_mTRF - z_ridge: m

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def _extract_arrays(df):
    eel = df["eel"].to_numpy(dtype=float)
    ridge = df["ridge"].to_numpy(dtype=float)
    mask = np.isfinite(eel) & np.isfinite(ridge)
    eel, ridge = eel[mask], ridge[mask]
    diff = eel - ridge
    return eel, ridge, diff

def plot_hist_and_paired_grid(df_phasic, df_tonic, df_combined,
                              xlim_diff=(-0.3, 0.3),
                              ylim_paired=(-0.2, 0.5),
                              bins=12,
                              figsize=(10, 5.5)):

    dfs = [df_phasic, df_tonic, df_combined]
    col_titles = ["Phasic", "Tonic", "Combined"]

    # --- extract arrays & compute a common hist y-limit ---
    data = []
    all_counts_max = 0
    bin_edges = np.linspace(xlim_diff[0], xlim_diff[1], bins + 1)

    for df in dfs:
        eel, ridge, diff = _extract_arrays(df)
        data.append((eel, ridge, diff))

        counts, _ = np.histogram(diff, bins=bin_edges)
        all_counts_max = max(all_counts_max, int(counts.max()) if counts.size else 0)

    ylim_hist = (0, max(4, all_counts_max + 2))

    # --- figure layout ---
    fig, axes = plt.subplots(2, 3, figsize=figsize, sharey="row")
    fig.subplots_adjust(
        left=0.07, right=0.99, bottom=0.10, top=0.88,
        wspace=0.12, hspace=0.32
    )

    # --- plotting ---
    for j, (title, (eel, ridge, diff)) in enumerate(zip(col_titles, data)):

        # ---------- Row 1: histogram ----------
        ax = axes[0, j]
        ax.hist(diff, bins=bin_edges, density=False, alpha=0.7)

        mean_d = float(np.mean(diff)) if diff.size else np.nan
        med_d  = float(np.median(diff)) if diff.size else np.nan

        ax.axvline(0.0, linestyle="--", linewidth=1.5)
        ax.axvline(mean_d, linestyle="-", linewidth=1.5,
                   label=f"mean Δr ", color="orange")
        ax.axvline(med_d, linestyle=":", linewidth=1.5,
                   label=f"median Δr", color="red")

        ax.set_xlim(*xlim_diff)
        ax.set_ylim(*ylim_hist)
        ax.set_title(title)

        ax.set_xlabel(r"$\Delta r^{test}$ (mTRF - ridge)")

        # fewer y-ticks on histogram
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))

        if j == 0:
            ax.set_ylabel("Count")
            ax.legend()
        else:
            ax.set_ylabel("")
            if ax.get_legend() is not None:
                ax.get_legend().remove()

        # ---------- Row 2: paired plot ----------
        ax = axes[1, j]
        n = diff.size
        x = np.arange(n)

        ax.plot(x, ridge, marker="o", linestyle="", label="Ridge")
        ax.plot(x, eel,   marker="s", linestyle="", label="mTRF")

        for i in range(n):
            color = "orange" if diff[i] > 0 else "gray"
            ax.plot([x[i], x[i]], [ridge[i], eel[i]],
                    color=color, linewidth=1.5, alpha=0.9)

        ax.axhline(0, linestyle="--", linewidth=1.5)
        ax.set_ylim(*ylim_paired)
        ax.set_xlabel("Subject")

        # fewer y-ticks on paired row too
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

        if j == 0:
            ax.set_ylabel(r"$r^{test}$")
            ax.legend()
        else:
            ax.set_ylabel("")
            if ax.get_legend() is not None:
                ax.get_legend().remove()

        ax.grid(True)

    plt.show()


# call:
plot_hist_and_paired_grid(df_phasic, df_tonic, df_combined)


In [10]:
import pandas as pd
import numpy as np
from scipy import stats

# Load both datasets
df_ridge = pd.read_csv(r'C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\ridge_reg\match_eelbrain_cv\dfk_ridge_eelbrain_ALL.csv')

# Basic statistics
print("\nComparison Statistics:")
print(f"Correlation between 'cv_mean_test_r_lam0' and 'cv_mean_test_r_star': {df_ridge['cv_mean_test_r_lam0'].corr(df_ridge['cv_mean_test_r_star']):.4f}")
print(f"\nDifference (cv_mean_test_r_lam0 - cv_mean_test_r_star):")
print((df_ridge['cv_mean_test_r_lam0'] - df_ridge['cv_mean_test_r_star']).describe())

# --- subject-wise diff ---
df_ridge = df_ridge.dropna(subset=["cv_mean_test_r_lam0", "cv_mean_test_r_star"]).copy()
df_ridge["diff"] = df_ridge["cv_mean_test_r_lam0"] - df_ridge["cv_mean_test_r_star"]

diff = df_ridge["diff"].to_numpy()

print("N subjects:", len(diff))
print("Mean diff (eelbrain - ridge):", diff.mean())

# --- Paired t-test (one-sided) ---
t_stat, p_one = stats.ttest_rel(df_ridge["cv_mean_test_r_lam0"], df_ridge["cv_mean_test_r_star"], alternative='less')
print("\nPaired t-test (one-sided):")
print("t =", t_stat, "p =", p_one)





Comparison Statistics:
Correlation between 'cv_mean_test_r_lam0' and 'cv_mean_test_r_star': 0.9214

Difference (cv_mean_test_r_lam0 - cv_mean_test_r_star):
count    52.000000
mean     -0.007862
std       0.019866
min      -0.065199
25%      -0.013463
50%      -0.004396
75%       0.002720
max       0.037690
dtype: float64
N subjects: 52
Mean diff (eelbrain - ridge): -0.007862485391581709

Paired t-test (one-sided):
t = -2.8539095908331222 p = 0.0031133209788268483
